In [2]:
import torch
import triton
import triton.language as tl

In [3]:
@triton.jit
def vector_add_kernel(a, b, c, n_elements, BLOCK_SIZE: tl.constexpr):
    pid=tl.program_id(axis=0)
    block_start=pid*BLOCK_SIZE
    block_solve_range=block_start+tl.arange(0,BLOCK_SIZE)
    mask=block_solve_range<n_elements
    x=tl.load(a+block_solve_range,mask=mask)
    y=tl.load(b+block_solve_range,mask=mask)
    output=x+y
    tl.store(c+block_solve_range,output,mask=mask)
# a, b, c are tensors on the GPU
def solve(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor, N: int):    
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(N, BLOCK_SIZE),)
    vector_add_kernel[grid](a, b, c, N, BLOCK_SIZE)

In [4]:
device=torch.device('cuda:0')
a=torch.randn(10000,device=device)
b=torch.randn(10000,device=device)
c=torch.empty_like(a,device=device)
solve(a,b,c,10000)

d:\Anaconda\envs\py312pt291cu128\Lib\site-packages\torch\cuda\__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce RTX 5070 Ti Laptop GPU which is of cuda capability 12.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (5.0) - (9.0)
    
  warnings.warn(
d:\Anaconda\envs\py312pt291cu128\Lib\site-packages\torch\cuda\__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.8 13.0 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
d:\Anaconda\envs\py312pt291cu128\Lib\site-packages\torch\cuda\__init__.py:326: UserWarning: 
NVIDIA GeForce RTX 5070 Ti Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5070 Ti Laptop GPU GPU with PyT

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [1]:
import torch
print(torch.cuda.is_available())

True


In [9]:
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
print(f"CUDA 版本: {torch.version.cuda}")
print(f"GPU 设备: {torch.cuda.get_device_name(0)}")

PyTorch 版本: 2.9.1+cu126
CUDA 是否可用: True
CUDA 版本: 12.6
GPU 设备: NVIDIA GeForce RTX 5070 Ti Laptop GPU
